# Part 1 — Data Analysis and Management with Hadoop and Hive
**Contribution:** Individual — Suman  
**Dataset:** Official bank.csv, 4,521 rows, 17 columns.

## Execution scope
HDFS runs a real NameNode and DataNode with replication factor 1. Python mapper/reducer programs run through the Hadoop Streaming JAR using LocalJobRunner over HDFS. HiveServer2 executes HiveQL against a typed ORC table in HDFS, using the local MapReduce engine. This laptop demonstration does not claim a YARN cluster or multi-machine fault tolerance.

The commands below reproduce the local run from the project root. Outputs are read from the saved execution transcripts, rather than being invented or presented as newly executed Hadoop jobs inside Colab. The accompanying Google Doc includes code/output screenshots under the official questions.


## HDFS ingestion and MapReduce execution
```powershell
docker compose --profile hadoop up -d hadoop-hive
docker compose --profile hadoop exec -T hadoop-hive bash /workspace/hadoop/run_mapreduce.sh
docker compose --profile hadoop exec -T hadoop-hive bash /workspace/hive/run_hive.sh
```


In [1]:
from pathlib import Path
root=Path.cwd()
if not (root/'data/bank.csv').exists(): root=root.parent
print('Dataset bytes:',(root/'data/bank.csv').stat().st_size)
print('Execution file:',root/'reports/hadoop/execution.log')


Dataset bytes: 375542
Execution file: /workspace/reports/hadoop/execution.log


## Hadoop Q1 Data ingestion
The HDFS report and file listing confirm that the original 375,542-byte CSV was transferred. The CSV header is ignored by the Python mapper. A single DataNode provides no replica failover.

In [2]:
print((root/'reports/hadoop/execution.log').read_text().split('++ date')[0])

+ hdfs dfs -mkdir -p /banking/raw
+ hdfs dfs -put -f /workspace/data/bank.csv /banking/raw/bank.csv
+ hdfs dfs -ls /banking/raw
Found 1 items
-rw-r--r--   1 root supergroup     375542 2026-10-20 18:46 /banking/raw/bank.csv
+ hdfs dfsadmin -report
Configured Capacity: 1081101176832 (1006.85 GB)
Present Capacity: 988887695360 (920.97 GB)
DFS Remaining: 988887277568 (920.97 GB)
DFS Used: 417792 (408 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 172.21.0.2:9866 (hadoop-hive)
Hostname: hadoop-hi

## Hadoop Q2 Python programs and results
Mappers emit keyed sums and counts; the reducer combines all values for each shuffled key. Mean balances use total balance divided by record count, avoiding an incorrect average of partition averages.

In [3]:
print((root/'hadoop/mapper.py').read_text());print((root/'hadoop/reducer.py').read_text())

#!/usr/bin/env python3
"""Hadoop Streaming mapper. Input is parsed as CSV, not split by commas manually."""
import csv
import sys
mode=sys.argv[1]
fields='age job marital education default balance housing loan contact day month duration campaign pdays previous poutcome y'.split()
for values in csv.reader(sys.stdin):
    if not values or values[0]=='age':continue
    if len(values)!=len(fields):
        raise ValueError('Invalid bank.csv record length')
    r=dict(zip(fields,values))
    if mode=='job_balance':key,value=r['job'],int(r['balance'])
    elif mode=='education_housing':key,value=r['education']+'|'+r['housing'],1
    elif mode=='month_subscription':key,value=r['month']+'|'+r['y'],1
    elif mode=='outcome_duration':key,value=r['poutcome'],int(r['duration'])
    elif mode=='age_balance':key,value=str(int(r['age'])//10*10)+'s',int(r['balance'])
    else:raise ValueError('Unknown mapper mode '+mode)
    print(f'{key}\t{value}\t1')

#!/usr/bin/env python3
import itertools
import 

### Average balance by occupation

In [4]:
print((root/'reports/hadoop/job_balance.tsv').read_text())

admin.	478	1226.736402
blue-collar	946	1085.161734
entrepreneur	168	1645.125000
housemaid	112	2083.803571
management	969	1766.928793
retired	230	2319.191304
self-employed	183	1392.409836
services	417	1103.956835
student	84	1543.821429
technician	768	1330.996094
unemployed	128	1089.421875
unknown	38	1501.710526



### Housing-loan counts by education

In [5]:
print((root/'reports/hadoop/education_housing.tsv').read_text())

primary|no	295
primary|yes	383
secondary|no	876
secondary|yes	1430
tertiary|no	687
tertiary|yes	663
unknown|no	104
unknown|yes	83



### Monthly contact counts by subscription status

In [6]:
print((root/'reports/hadoop/month_subscription.tsv').read_text())

apr|no	237
apr|yes	56
aug|no	554
aug|yes	79
dec|no	11
dec|yes	9
feb|no	184
feb|yes	38
jan|no	132
jan|yes	16
jul|no	645
jul|yes	61
jun|no	476
jun|yes	55
mar|no	28
mar|yes	21
may|no	1305
may|yes	93
nov|no	350
nov|yes	39
oct|no	43
oct|yes	37
sep|no	35
sep|yes	17



### Hadoop Q3 Average call duration by previous outcome

In [7]:
print((root/'reports/hadoop/outcome_duration.tsv').read_text())

failure	490	254.383673
other	197	273.832487
success	129	338.635659
unknown	3705	262.103104



### Hadoop Q3 Age decade and average balance

In [8]:
print((root/'reports/hadoop/age_balance.tsv').read_text())

10s	4	393.500000
20s	478	1063.656904
30s	1808	1298.147124
40s	1203	1474.692436
50s	854	1547.420375
60s	113	2619.778761
70s	47	3280.872340
80s	14	2071.142857



## Hive Q1 to Q9 SQL and execution evidence
The original CSV is staged using OpenCSVSerde, then explicitly cast into a typed ORC table named banking_data.client_info. A defensive age/label filter excludes the CSV header even if header-skipping is bypassed by the execution path. The verified table contains 4,521 rows.

Q7 cannot be calculated because no year or full date exists. Q8 analyzes the supplied average yearly balance field across education groups; it is a cross-sectional comparison, not a year-over-year time series. A three-standard-deviation heuristic flags unusual within-group balances for inspection and is not a fraud label.


In [9]:
print((root/'hive/queries.sql').read_text())

SET hive.execution.engine=mr;
SET hive.exec.mode.local.auto=true;
SET hive.auto.convert.join=false;
CREATE DATABASE IF NOT EXISTS banking_data;
USE banking_data;

-- Q1: quote-aware CSV staging, followed by a typed ORC client_info table.
DROP TABLE IF EXISTS raw_bank;
CREATE EXTERNAL TABLE raw_bank (
 age STRING, job STRING, marital STRING, education STRING, `default` STRING,
 balance STRING, housing STRING, loan STRING, contact STRING, day STRING,
 month STRING, duration STRING, campaign STRING, pdays STRING, previous STRING,
 poutcome STRING, y STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ('separatorChar'=',', 'quoteChar'='"')
STORED AS TEXTFILE LOCATION '/banking/raw'
TBLPROPERTIES ('skip.header.line.count'='1');
DROP TABLE IF EXISTS client_info;
CREATE TABLE client_info STORED AS ORC AS SELECT
 CAST(age AS INT) age, job, marital, education, `default`,
 CAST(balance AS INT) balance, housing, loan, contact, CAST(day AS INT) day,
 month, 

In [10]:
print((root/'reports/hive-results.txt').read_text())

col_name	data_type	comment
age	int	
job	string	
marital	string	
education	string	
default	string	
balance	int	
housing	string	
loan	string	
contact	string	
day	int	
month	string	
duration	int	
campaign	int	
pdays	int	
previous	int	
poutcome	string	
y	string	
question	clients
Q2_count	4521
client_info.age	client_info.job	client_info.marital	client_info.education	client_info.default	client_info.balance	client_info.housing	client_info.loan	client_info.contact	client_info.day	client_info.month	client_info.duration	client_info.campaign	client_info.pdays	client_info.previous	client_info.poutcome	client_info.y
30	unemployed	married	primary	no	1787	no	no	cellular	19	oct	79	1	-1	0	unknown	no
33	services	married	secondary	no	4789	yes	yes	cellular	11	may	220	1	339	4	failure	no
35	management	single	tertiary	no	1350	yes	no	cellular	16	apr	185	1	330	1	failure	no
30	management	married	tertiary	no	1476	yes	yes	unknown	3	jun	199	4	-1	0	unknown	no
59	blue-collar	married	secondary	no	0	yes	no	unknown	5	m

## Findings and interpretation
May has 1,398 recorded contacts and a subscription rate of 6.65%. Retired clients have the highest mean balance by occupation at about 2,319.19, while group sizes vary substantially. Pearson age/balance correlation is approximately 0.0838, indicating weak linear association. Previously successful campaigns are associated with higher current subscription rates, but this does not establish a causal effect.

The unusual balance heuristic flags 53 secondary, 35 tertiary, 6 primary and 4 unknown-education records. Negative balances are possible and retained. These flags identify records for data review, not proof of fraud. Source data lacks currency metadata, so no currency symbol is invented.
